# PA4: Simple Classifiers

- Programmer: Lydia Lonzarich
- Class: CPSC 322-01, Fall 2025
- Programming Assignment #4
- Date of current version: 10/25/2025
- Description: this notebook employs three simple classifiers (linear regression, kNN, and a dummy), training them on train sets, and evaluating their performance on a test set with 5 instances.

In [2]:
# some useful mysklearn package import statements and reloads
import importlib

import mysklearn.myutils
importlib.reload(mysklearn.myutils)
import mysklearn.myutils as myutils
import numpy as np
from mysklearn.myutils import mpg_discretizer

import mysklearn.mypytable
importlib.reload(mysklearn.mypytable)
from mysklearn.mypytable import MyPyTable 

import mysklearn.myclassifiers
importlib.reload(mysklearn.myclassifiers)
from mysklearn.myclassifiers import MySimpleLinearRegressionClassifier,\
    MyKNeighborsClassifier,\
    MyDummyClassifier

# Step 0: Train / Test Sets with Random Instances
In this step, I create the training and testing sets that will be used for each of the following steps.

In [124]:
# load the dataset.
filename = "auto-data-removed-NA.txt"
pytable = MyPyTable()
pytable.load_from_file(filename)

# convert all integer values to floats.
pytable.convert_to_numeric()

data = np.array(pytable.data, dtype=object) # use dtype=object to preserve all instance's data types.

# seed random number generator.
np.random.seed(100)

# "find" and shuffle indices.
indices = np.arange(len(data))
np.random.shuffle(indices)

# select 5 instances from the dataset to form the test set. (pull from the first 5 indices in 'indices').
test_indices = indices[:5]
test = data[test_indices]

# use the remaining instances from the dataset to form the train set. (pull from index 5 to the last index in 'indices')
train_indices = indices[5:]
train = data[train_indices]

# Step 1: Train / Test Sets with Random Instances and Linear Regression
In this step, I create a simple linear regression classier to predict DOE mpg ratings using vehicle weight. 

In [125]:
# find indices of the 'weight' and 'mpg' column in the table.
weight_indices = pytable.column_names.index("weight")
mpg_indices = pytable.column_names.index("mpg")

# create X_train and y_train: take 1 or more instances from the train set.
X_train = train[:, weight_indices]
y_train = train[:, mpg_indices]

# create X_test with the 5 instances in the test set from step (0).
X_test = test[:, weight_indices]
y_test = test[:, mpg_indices]

# make X_train and X_test into 2D lists
X_train = [[i] for i in X_train]
X_test = [[i] for i in X_test]

# create a linear regression object.
my_linreg = MySimpleLinearRegressionClassifier(discretizer=mpg_discretizer)

# train the linear regression model object on the training data. 
my_linreg.fit(X_train, y_train)

# predict MPG for the test instances.
y_pred = my_linreg.predict(X_test)

pred_ratings = y_pred

# convert the continuous true mpg values to a categorical rating using the discretizer.
actual_ratings = [my_linreg.discretizer(y) for y in y_test]

print("=============================================")
print("STEP 1: Linear Regression MPG Classifier")
print("=============================================")
count = 0 # to keep track of correct predictions.

# iterate over each test instance and compare its predicted mpg rating to its actual mpg rating.
for instance in range(len(test)):
    pred = pred_ratings[instance]
    actual = actual_ratings[instance]
    if pred == actual:
        count += 1

    # display results.
    print("Instance: ", test[instance])
    print("Class: ", pred, "Actual: ", actual)

# report the accuracy of the classifier for the 5 test instances. 
acc = count / len(X_test)
print("Accuracy of simple linear regression classifier: ", acc)



STEP 1: Linear Regression MPG Classifier
Instance:  [25.0 4.0 121.0 115.0 2671.0 13.5 75.0 2.0 'saab 99le' 5198.0]
Class:  6 Actual:  6
Instance:  [19.0 4.0 121.0 112.0 2868.0 15.5 73.0 2.0 'volvo 144ea' 4340.0]
Class:  5 Actual:  4
Instance:  [31.5 4.0 98.0 68.0 2045.0 18.5 77.0 3.0 'honda accord cvcc' 3299.0]
Class:  7 Actual:  8
Instance:  [23.2 4.0 156.0 105.0 2745.0 16.7 78.0 1.0 'plymouth sapporo' 6087.0]
Class:  6 Actual:  6
Instance:  [20.2 6.0 232.0 90.0 3265.0 18.2 79.0 1.0 'amc concord dl 6' 4324.0]
Class:  5 Actual:  5
Accuracy of simple linear regression classifier:  0.6


# Step 2: Train / Test Sets with Random Instances and kNN
In this step, I create a nearest neighbor classifier that predicts ODE mpg ratings using the number of cylinder, weight, and accleration attributes for k = 5. 

In [116]:
# find indices of the 'cylinders', 'weight', 'acceleration', and 'mpg' column in the table.
cylinder_indices = pytable.column_names.index("cylinders")
weight_indices = pytable.column_names.index("weight")
acceleration_indices = pytable.column_names.index("acceleration")
mpg_indices = pytable.column_names.index("mpg")

# create X_train and y_train: take 1 or more instances from the train set.
X_train = np.column_stack((train[:, cylinder_indices], train[:, weight_indices], train[:, acceleration_indices]))
y_train = train[:, mpg_indices]

# create X_test with the 5 instances in the test set from step (0).
X_test = np.column_stack((test[:, cylinder_indices], test[:, weight_indices], test[:, acceleration_indices]))
y_test = test[:, mpg_indices]

# normalize features in X_train and X_test using min-max normalization.
min_train = X_train.min() 
max_train = X_train.max()
X_train = (X_train - min_train) / (max_train - min_train)
X_test = (X_test - min_train) / (max_train - min_train)

# create a nearest neighbor classifier object.
my_knn = MyKNeighborsClassifier(n_neighbors=5)

# train the knn object on the train data.
my_knn.fit(X_train, y_train)

# predict MPG for the test instances.
y_pred = my_knn.predict(X_test)

pred_ratings = [mpg_discretizer(y) for y in y_pred]

# convert the continuous true mpg values to a categorical rating using the discretizer.
actual_ratings = [mpg_discretizer(y) for y in y_test]


print("=============================================")
print("STEP 2: k=5 Nearest Neighbor MPG Classifier")
print("=============================================")
count = 0 # to keep track of correct predictions.

# iterate over each test instance and compare its predicted mpg rating to its actual mpg rating.
for instance_idx in range(len(test)):
    pred = pred_ratings[instance_idx] # get the "value" (aka, prediction) by grabbing it by its idx from the list with all predictions in it.
    actual = actual_ratings[instance_idx]
    if pred == actual:
        count += 1

    # display results.
    print("Instance: ", test[instance_idx])
    print("Class: ", pred, "Actual: ", actual)

# report the accuracy of the classifier for the 5 test instances. 
acc = count / len(X_test)
print("Accuracy of the nearest neighbor classifier: ", acc)


STEP 2: k=5 Nearest Neighbor MPG Classifier
Instance:  [25.0 4.0 121.0 115.0 2671.0 13.5 75.0 2.0 'saab 99le' 5198.0]
Class:  7 Actual:  6
Instance:  [19.0 4.0 121.0 112.0 2868.0 15.5 73.0 2.0 'volvo 144ea' 4340.0]
Class:  6 Actual:  4
Instance:  [31.5 4.0 98.0 68.0 2045.0 18.5 77.0 3.0 'honda accord cvcc' 3299.0]
Class:  7 Actual:  8
Instance:  [23.2 4.0 156.0 105.0 2745.0 16.7 78.0 1.0 'plymouth sapporo' 6087.0]
Class:  6 Actual:  6
Instance:  [20.2 6.0 232.0 90.0 3265.0 18.2 79.0 1.0 'amc concord dl 6' 4324.0]
Class:  4 Actual:  5
Accuracy of the nearest neighbor classifier:  0.2


# Step 3: Train / Test Sets with Random Instances and Dummy Classification
In this step, I create a dummy classifier to predict DOE mpg ratings. 

In [117]:
# find indices of the 'weight' and 'mpg' column in the table.
weight_indices = pytable.column_names.index("weight")
mpg_indices = pytable.column_names.index("mpg")

# create X_train and y_train: take 1 or more instances from the train set.
X_train = np.column_stack((train[:, cylinder_indices], train[:, weight_indices], train[:, acceleration_indices]))
y_train = train[:, mpg_indices]

# print(X_train)
# print(X_train.shape)
# print(type(X_train[0,0]))

# create X_test with the 5 instances in the test set from step (0).
X_test = np.column_stack((test[:, cylinder_indices], test[:, weight_indices], test[:, acceleration_indices]))
y_test = test[:, mpg_indices]

# create a dummy classifier object.
my_dummy = MyDummyClassifier()

# train the dummy object on the train data.
my_dummy.fit(X_train, y_train)

# predict MPG for the test instances.
y_pred = my_dummy.predict(X_test)

pred_ratings = [mpg_discretizer(y) for y in y_pred]

# convert the continuous true mpg values to a categorical rating using the discretizer.
actual_ratings = [mpg_discretizer(y) for y in y_test]


print("=============================================")
print("STEP 3: (Zero-R) Dummy MPG Classifier")
print("=============================================")
count = 0 # to keep track of correct predictions.

# iterate over each test instance and compare its predicted mpg rating to its actual mpg rating.
for instance_idx in range(len(test)):
    pred = pred_ratings[instance_idx]
    actual = actual_ratings[instance_idx]
    if pred == actual:
        count += 1

    # display results.
    print("Instance: ", test[instance_idx])
    print("Class: ", pred, "Actual: ", actual)

# report the accuracy of the classifier for the 5 test instances. 
acc = count / len(X_test)
print("Accuracy of (Zero-R) Dummy Classifier: ", acc)


STEP 3: (Zero-R) Dummy MPG Classifier
Instance:  [25.0 4.0 121.0 115.0 2671.0 13.5 75.0 2.0 'saab 99le' 5198.0]
Class:  1 Actual:  6
Instance:  [19.0 4.0 121.0 112.0 2868.0 15.5 73.0 2.0 'volvo 144ea' 4340.0]
Class:  1 Actual:  4
Instance:  [31.5 4.0 98.0 68.0 2045.0 18.5 77.0 3.0 'honda accord cvcc' 3299.0]
Class:  1 Actual:  8
Instance:  [23.2 4.0 156.0 105.0 2745.0 16.7 78.0 1.0 'plymouth sapporo' 6087.0]
Class:  1 Actual:  6
Instance:  [20.2 6.0 232.0 90.0 3265.0 18.2 79.0 1.0 'amc concord dl 6' 4324.0]
Class:  1 Actual:  5
Accuracy of (Zero-R) Dummy Classifier:  0.0


# Step 4: Classifier Comparison: Linear Regression vs kNN vs Dummy
- The simple linear regression classifier performed best among the 3 classifiers I implemented and tested in this programming assignment.
- I tested the performance of my model with 4 different different random seeds (0, 1, 42, 100, 123) to create observe the performance of my models with 4 distinct train/test sets. 
- I found that the simple linear regressor performed best, with accuracies of 0.8, 0.2, 0.4, and 0.6, respectively. 
- I found that the kNN classifier produced an accuracy of either 0.0 or 0.2, which means that the model was predicting 0 or 1 of the 5 test labels correctly. This behavior is expected with a k=5 in a small dataset because we are only looking at 5 other instances to generate predictions.
- Lastly, I found that the dummy classifier performed the worst, as it always produced an accuracy of 0.0.
- With such a small test set, this behavior of these models is expected. 
- To improve the reliability of my comparisons, I could have created a list of integer values that I want to use to set my random seed, and iterate through each model and output its accuracy. 